# FactorFolio Step 1: Offline Training Notebook

**AC420 project, Jasper Lin.** This notebook runs once, offline, in Google Colab. It turns course data into small result files that the Streamlit app reads.

**What it makes (saved to the repo's `data/` folder):**
* `sp100_estimates.csv`: CAPM and Fama-French 3-factor estimates for each S&P 100 stock in the class data. These are the reference values that scale the app's 0 to 100 risk score.
* `market_scenarios.csv`: bear, median and bull annual market returns (1927 to 2025) for the app's projections.
* `ff_factors_monthly.csv`: the monthly Fama-French factors the live app uses for its regressions.

**Data used:**
* `stock_returns.csv` and `FF_factors.csv` from the Lecture 17 folder (WRDS). Course rule: WRDS data is only used here, for training. It is never uploaded to the app or the GitHub repo.
* `sp100_tickers.csv`: S&P 100 tickers with sector and industry from yfinance.
* `F-F_Research_Data_Factors.csv`: free public factors from Ken French's Data Library.

**Code sources:** the Lecture 12 & 13 and Lecture 17 regression notebooks, plus the Lecture 17 study guide's year-month merge fix. Written with AI assistance (Claude); see my AI usage log.

**To run in Colab:** upload the four CSV files above (folder icon on the left, then the upload button), then Runtime, then Run all.

### Import libraries and set the data folder

In [ ]:
import pandas as pd
import statsmodels.api as sm

DATA_DIR = '/content/'   # Colab's upload folder

### Step A: Load the Fama-French monthly factors (Lecture 17)
* Look at the data first: factor returns should be decimals (0.0281 = 2.81%).

In [ ]:
factors = pd.read_csv(DATA_DIR + 'FF_factors.csv')
factors = factors.rename(columns={'dateff': 'datadate'})
print(factors.shape)
factors.head()

### Step B: Load the WRDS monthly stock returns (Lecture 17)
* Keep only the columns we need, drop missing returns, and require at least 50 months per stock.
* `trt1m` is in percent, so divide by 100 to match the factors.

In [ ]:
returns = pd.read_csv(DATA_DIR + 'stock_returns.csv', usecols=['datadate', 'tic', 'trt1m'])
returns = returns.dropna(subset=['trt1m'])
returns = returns[returns.groupby('tic')['tic'].transform('size') >= 50]
returns['ret'] = returns['trt1m'] / 100
print(returns.shape, returns['tic'].nunique(), 'tickers')
returns['ret'].describe()

### Step C: Merge on year-month, then recount (Lecture 17 fix)
* The factor file dates each month by its last trading day; the WRDS file uses the calendar month-end.
* Merging on the exact date silently drops months, so merge on year-month instead and compare row counts.

In [ ]:
returns['ym'] = pd.to_datetime(returns['datadate']).dt.to_period('M')
factors['ym'] = pd.to_datetime(factors['datadate']).dt.to_period('M')
combined = pd.merge(returns, factors.drop(columns='datadate'), on='ym', how='inner')
print('rows before merge:', len(returns), ' rows after merge:', len(combined))

### Step D: CAPM and Fama-French 3-factor regressions for each S&P 100 stock (Lectures 12 to 17)
* CAPM: (R - Rf) = alpha + beta(Mkt - Rf) + e
* FF3: (R - Rf) = alpha + b1(Mkt - Rf) + b2(SMB) + b3(HML) + e
* Yahoo writes class shares with a dash (BRK-B) and WRDS with a dot (BRK.B), so the ticker is converted before matching.

In [ ]:
sp100 = pd.read_csv(DATA_DIR + 'sp100_tickers.csv')

rows = []
for _, stock in sp100.iterrows():
    tic = stock['ticker'].replace('-', '.')
    tic_df = combined[combined['tic'] == tic].copy()
    if tic_df.empty:
        print('not in class file:', stock['ticker'])
        continue
    y = tic_df['ret'] - tic_df['rf']                                   # excess return
    capm = sm.OLS(y, sm.add_constant(tic_df['mktrf']), missing='drop').fit()
    ff3 = sm.OLS(y, sm.add_constant(tic_df[['mktrf', 'smb', 'hml']]), missing='drop').fit()
    rows.append({'ticker': stock['ticker'], 'sector': stock['sector'], 'industry': stock['industry'],
                 'n_months': int(capm.nobs),
                 'alpha': capm.params['const'], 'alpha_p': capm.pvalues['const'],
                 'beta': capm.params['mktrf'], 'r2': capm.rsquared,
                 'ff3_alpha': ff3.params['const'], 'ff3_alpha_p': ff3.pvalues['const'],
                 'ff3_beta': ff3.params['mktrf'], 's': ff3.params['smb'], 'h': ff3.params['hml'],
                 'ff3_r2': ff3.rsquared, 'ff3_adj_r2': ff3.rsquared_adj,
                 'volatility': tic_df['ret'].std(),
                 'firm_specific': 1 - capm.rsquared})

estimates = pd.DataFrame(rows)
print(len(estimates), 'of', len(sp100), 'S&P 100 stocks estimated')
estimates.describe()

### Known-answer check (Lecture 17 study guide)
After the year-month fix, MSFT should show 64 months, CAPM beta 1.064 and FF3 R-squared 0.614.

In [ ]:
msft = estimates[estimates['ticker'] == 'MSFT'].iloc[0]
print('MSFT months:', msft['n_months'], ' beta:', round(msft['beta'], 3), ' FF3 R2:', round(msft['ff3_r2'], 3))
assert msft['n_months'] == 64 and abs(msft['beta'] - 1.064) < 0.001 and abs(msft['ff3_r2'] - 0.614) < 0.001

### Step E: Market scenarios and monthly factors from Ken French's Data Library
* The file has a monthly section, then an annual section. Both are in percent.
* Annual market return = (Mkt-RF + RF) / 100. The bear, median and bull cases are the 25th, 50th and 75th percentiles from `describe()`.
* The 10th and 90th percentiles are saved too, in case the app's bear case should be a true down year (the 25th percentile is close to 0%).

In [ ]:
ff_lines = open(DATA_DIR + 'F-F_Research_Data_Factors.csv').read().splitlines()
annual_start = next(i for i, line in enumerate(ff_lines) if 'Annual' in line)

def read_ff_section(lines):
    data = []
    for line in lines:
        parts = [p.strip() for p in line.split(',')]
        if len(parts) == 5 and parts[0].isdigit():
            data.append([int(parts[0])] + [float(p) for p in parts[1:]])
    return pd.DataFrame(data, columns=['period', 'mktrf', 'smb', 'hml', 'rf'])

monthly_ff = read_ff_section(ff_lines[:annual_start])
annual_ff = read_ff_section(ff_lines[annual_start:])
print('monthly:', monthly_ff['period'].min(), 'to', monthly_ff['period'].max(), len(monthly_ff), 'rows')
print('annual:', annual_ff['period'].min(), 'to', annual_ff['period'].max(), len(annual_ff), 'rows')

annual_ff['market_return'] = (annual_ff['mktrf'] + annual_ff['rf']) / 100
annual_ff['market_return'].describe(percentiles=[.10, .25, .50, .75, .90])

In [ ]:
stats = annual_ff['market_return'].describe(percentiles=[.10, .25, .50, .75, .90])
first, last = annual_ff['period'].min(), annual_ff['period'].max()
src = f'annual U.S. market returns {first} to {last}, Ken French Data Library'
scenarios = pd.DataFrame([
    {'name': 'bear_market_return',   'value': stats['25%'],  'description': '25th percentile of ' + src},
    {'name': 'median_market_return', 'value': stats['50%'],  'description': 'Median of ' + src},
    {'name': 'bull_market_return',   'value': stats['75%'],  'description': '75th percentile of ' + src},
    {'name': 'mean_market_return',   'value': stats['mean'], 'description': 'Average of ' + src},
    {'name': 'severe_bear_market_return', 'value': stats['10%'], 'description': '10th percentile of ' + src},
    {'name': 'strong_bull_market_return', 'value': stats['90%'], 'description': '90th percentile of ' + src},
    {'name': 'avg_smb', 'value': annual_ff['smb'].mean() / 100, 'description': f'Average annual SMB {first} to {last}'},
    {'name': 'avg_hml', 'value': annual_ff['hml'].mean() / 100, 'description': f'Average annual HML {first} to {last}'},
])
scenarios

### Step F: Save the result files
* Only estimated numbers and public Ken French data are saved. No raw WRDS data leaves this notebook.
* `ff_factors_monthly.csv` stays in percent, as Ken French posts it; the app divides by 100 (data check 4).

In [ ]:
monthly_ff = monthly_ff.rename(columns={'period': 'yyyymm'})
estimates.to_csv('sp100_estimates.csv', index=False)
scenarios.to_csv('market_scenarios.csv', index=False)
monthly_ff.to_csv('ff_factors_monthly.csv', index=False)

try:
    from google.colab import files     # only works in Colab
    for f in ['sp100_estimates.csv', 'market_scenarios.csv', 'ff_factors_monthly.csv']:
        files.download(f)
except ImportError:
    print('Saved files locally (not running in Colab).')